# CÂU 1 - XỬ LÝ HISTOGRAM ẢNH

Notebook thực hiện:

- Chuyển ảnh màu sang ảnh xám.
- Tính Histogram gốc (H1).
- Histogram Equalization.
- Hiệu chỉnh mức xám về khoảng (30,120).
- Lưu toàn bộ kết quả cho nhiều ảnh.

## Bước 1: Import thư viện

Nạp các thư viện cần thiết cho xử lý ảnh, tính toán số học và hiển thị kết quả.

In [1]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Bước 2: Khai báo thư mục dữ liệu

Xác định thư mục chứa ảnh đầu vào và thư mục lưu kết quả đầu ra.
Nếu thư mục đầu ra chưa tồn tại thì tự động tạo mới.

In [ ]:
input_dir = "anh_xlas/anh_xlas"
output_dir = "output_cau1"

os.makedirs(output_dir, exist_ok=True)

image_files = sorted(glob.glob(os.path.join(input_dir, "*.jpg")))

print(f"Found {len(image_files)} images")

## Bước 3: Đọc toàn bộ ảnh

Tự động tìm và nạp tất cả ảnh JPG trong thư mục đầu vào.

In [ ]:
images = []

for image_path in image_files:
    img = cv2.imread(image_path)
    images.append((image_path, img))

print("Images loaded successfully.")

## Bước 4: Chuyển ảnh màu sang ảnh xám

Chuyển đổi ảnh RGB/BGR sang ảnh mức xám để thuận tiện cho việc xử lý Histogram.

In [ ]:
gray_images = []

for image_path, img in images:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_images.append((image_path, img, gray))

print("Gray conversion completed.")

## Bước 5: Tính Histogram gốc (H1)

Đếm số lượng điểm ảnh ở từng mức xám từ 0 đến 255.

In [ ]:
results = []

for image_path, img, gray in gray_images:

    H1 = np.zeros(256)

    H, W = gray.shape

    for y in range(H):
        for x in range(W):
            H1[gray[y, x]] += 1

    results.append({
        "path": image_path,
        "img": img,
        "gray": gray,
        "H1": H1
    })

print("Histogram H1 completed.")

## Bước 6: Cân bằng Histogram (H2)

Thực hiện Histogram Equalization để cải thiện độ tương phản của ảnh.

In [ ]:
for data in results:

    gray = data["gray"]
    H1 = data["H1"]

    H, W = gray.shape

    pdf = H1 / (H * W)

    cdf = np.cumsum(pdf)

    transform = np.round(cdf * 255).astype(np.uint8)

    equalized = transform[gray]

    H2 = np.zeros(256)

    for y in range(H):
        for x in range(W):
            H2[equalized[y, x]] += 1

    data["equalized"] = equalized
    data["H2"] = H2

print("Histogram Equalization completed.")

## Bước 7: Hiệu chỉnh Histogram về khoảng (30,120)

Ánh xạ các mức xám của ảnh sau cân bằng Histogram vào khoảng [30,120].

In [ ]:
for data in results:

    equalized = data["equalized"]

    stretch = np.round(
        30 + (equalized / 255.0) * (120 - 30)
    ).astype(np.uint8)

    H3 = np.zeros(256)

    H, W = stretch.shape

    for y in range(H):
        for x in range(W):
            H3[stretch[y, x]] += 1

    data["stretch"] = stretch
    data["H3"] = H3

print("Stretching completed.")

## Bước 8: Hiển thị kết quả

Hiển thị ảnh gốc, ảnh xám, Histogram gốc, ảnh cân bằng Histogram và Histogram sau hiệu chỉnh.

In [ ]:
for data in results:

    filename = os.path.basename(data["path"])

    plt.figure(figsize=(18,10))

    plt.subplot(2,4,1)
    plt.imshow(cv2.cvtColor(data["img"], cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(2,4,2)
    plt.imshow(data["gray"], cmap="gray")
    plt.title("Gray")
    plt.axis("off")

    plt.subplot(2,4,3)
    plt.plot(data["H1"])
    plt.title("Histogram H1")

    plt.subplot(2,4,4)
    plt.imshow(data["equalized"], cmap="gray")
    plt.title("Equalized")
    plt.axis("off")

    plt.subplot(2,4,5)
    plt.plot(data["H2"])
    plt.title("Histogram H2")

    plt.subplot(2,4,6)
    plt.imshow(data["stretch"], cmap="gray")
    plt.title("Stretch 30-120")
    plt.axis("off")

    plt.subplot(2,4,7)
    plt.plot(data["H3"])
    plt.title("Stretch Histogram")

    plt.tight_layout()
    plt.show()

## Bước 9: Lưu kết quả

Lưu toàn bộ ảnh và Histogram vào thư mục output_cau1 với tên tương ứng theo ảnh đầu vào.

In [ ]:
count = 0

for data in results:

    image_path = data["path"]

    filename = os.path.basename(image_path)
    name = os.path.splitext(filename)[0]

    cv2.imwrite(
        os.path.join(output_dir, f"Gray_{name}.jpg"),
        data["gray"]
    )

    cv2.imwrite(
        os.path.join(output_dir, f"Equalized_{name}.jpg"),
        data["equalized"]
    )

    cv2.imwrite(
        os.path.join(output_dir, f"Stretch_30_120_{name}.jpg"),
        data["stretch"]
    )

    plt.figure()
    plt.plot(data["H1"])
    plt.title("Histogram H1")
    plt.savefig(
        os.path.join(output_dir,
                     f"Histogram_{name}.png")
    )
    plt.close()

    plt.figure()
    plt.plot(data["H2"])
    plt.title("Equalized Histogram")
    plt.savefig(
        os.path.join(output_dir,
                     f"Equalized_Histogram_{name}.png")
    )
    plt.close()

    plt.figure()
    plt.plot(data["H3"])
    plt.title("Stretch Histogram")
    plt.savefig(
        os.path.join(output_dir,
                     f"Stretch_Histogram_{name}.png")
    )
    plt.close()

    print(f"Processing {filename} ... Done!")

    count += 1

print(f"Finished processing {count} images.")